In [1]:
import json
import time
from uuid import uuid4

import requests
from kafka import KafkaConsumer, KafkaProducer

producer = KafkaProducer(bootstrap_servers='kafka:9092')
consumer = KafkaConsumer(bootstrap_servers="kafka:9092")

producer.send('weather_stream', b'Bonjour Kafka')
producer.flush()

print(consumer.topics())
consumer.close()

{'weather_stream'}


In [2]:
consumer = KafkaConsumer(
    "weather_stream",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda value: value.decode("utf-8"),
    consumer_timeout_ms=5000,  # s'arrête après 5 secondes sans nouveau message
)

for message in consumer:
    print(message.value)

consumer.close()

Bonjour Kafka


In [3]:
cities = [
    {"city": "Paris", "country": "France", "latitude": 48.8566, "longitude": 2.3522},
    {"city": "Seville", "country": "Spain", "latitude": 37.3891, "longitude": -5.9845},
    {"city": "Dubai", "country": "United Arab Emirates", "latitude": 25.2048, "longitude": 55.2708},
    {"city": "Reykjavik", "country": "Iceland", "latitude": 64.1466, "longitude": -21.9426},
    {"city": "Wellington", "country": "New Zealand", "latitude": -41.2865, "longitude": 174.7762},
]

weather_producer = KafkaProducer(bootstrap_servers="kafka:9092")
api_url = "https://api.open-meteo.com/v1/forecast"
interval_seconds = 30

print("Envoi météo démarré. Interromps la cellule pour arrêter.")
try:
    while True:
        cycle_start = time.monotonic()
        for city_info in cities:
            try:
                response = requests.get(
                    api_url,
                    params={
                        "latitude": city_info["latitude"],
                        "longitude": city_info["longitude"],
                        "current": "temperature_2m,wind_speed_10m,weather_code",
                        "wind_speed_unit": "ms",
                    },
                    timeout=10,
                )
                response.raise_for_status()
                weather_data = response.json()
                weather_data.update(
                    city=city_info["city"],
                    country=city_info["country"],
                )
                weather_producer.send(
                    "weather_stream",
                    json.dumps(weather_data).encode("utf-8"),
                )
                print(json.dumps(weather_data, ensure_ascii=False))
            except requests.RequestException as error:
                print(f"Erreur météo pour {city_info['city']}: {error}")

        weather_producer.flush()
        elapsed = time.monotonic() - cycle_start
        time.sleep(max(0, interval_seconds - elapsed))
except KeyboardInterrupt:
    print("Envoi météo arrêté.")
finally:
    weather_producer.flush()
    weather_producer.close()

Envoi météo démarré. Interromps la cellule pour arrêter.
{"latitude": 48.853382, "longitude": 2.355208, "generationtime_ms": 172.35994338989258, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 36.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "°C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T15:30", "interval": 900, "temperature_2m": 12.5, "wind_speed_10m": 1.72, "weather_code": 3}, "city": "Paris", "country": "France"}
{"latitude": 37.364082, "longitude": -5.956622, "generationtime_ms": 34.98268127441406, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 18.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "°C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T15:30", "interval": 900, "temperature_2m": 20.6, "wind_speed_10m": 5.24, "weather_code": 80}, "city": "Sev